Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [ ]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U openai

# Importar bibliotecas.
import pandas as pd
import os
from google.colab import userdata
from openai import OpenAI

In [ ]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

Configurar modelo do groq a ser usado.

In [ ]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave previamente criada no Colnsole do Groq, site consele.groq.com
groq_api_key = userdata.get('groq_api_key')

# Inicializar o cliente da API.
client_ai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

# Modelo gpt-oss de 120 bilhões de parâmetros.
model_id = 'openai/gpt-oss-120b'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gpt_responses = []

# Repetição para percorrer todo Dataframe.
for index, row in df_questions_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    # Preparação do prompt em Markdown
    prompt_usuario = f"""
    ### PAPEL:
    {papel}

    ### ÁREA DE ATUAÇÃO:
    {categoria}

    ### CONTEXTO:
    {contexto}

    ### INSTRUÇÃO:
    {instrucao}
    """
    #completion = client_ai.chat.completions.create(
    response = client_ai.responses.create(
        model= model_id,
        messages=[
          # Por questões de sintaxes informo a role, pois é um campo obrigatório,
          # porém o conteúdo é somente o do Markdown.
          {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador.
    )
    response = response.output_text

    # Armazenar o resultado em uma lista.
    gpt_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA (somente se você a usou ativamente)
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gpt_response = pd.DataFrame(gpt_responses)

In [ ]:
df_questions_and_guidelines.head(1)